In [1]:
import os

In [2]:
os.chdir("../")

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [ ]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        return data_ingestion_config

In [ ]:
import urllib.request as request
import zipfile
from textSummarizer.logging import logger
from textSummarizer.utils.common import get_size

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):  # -> str:
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL, 
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} downloaded with headers: \n {headers}")
            # logger.info(f"Downloading file from: {self.config.source_URL} to {self.config.local_data_file}")
            # request.urlretrieve(self.config.source_URL, self.config.local_data_file)
            # logger.info(f"Downloaded file size: {get_size(self.config.local_data_file)}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")
            # logger.info(f"File already exists at {self.config.local_data_file}")

        # return self.config.local_data_file

    def extract_zip_file(self):  # -> str:
        """
        zip_file_path: str
        Extracts the zip file to the specified directory.
        Function returns None.
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        
        # if not os.path.exists(self.config.unzip_dir):
        #     logger.info(f"Extracting zip file to: {self.config.unzip_dir}")
        #     with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
        #         zip_ref.extractall(self.config.unzip_dir)
        #     logger.info(f"Extracted files to: {self.config.unzip_dir}")
        # else:
        #     logger.info(f"Unzip directory already exists at {self.config.unzip_dir}")

        # return self.config.unzip_dir


In [6]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2025-05-03 08:52:24,312 : INFO : common : YAML file config\config.yaml loaded successfully.]
[2025-05-03 08:52:24,316 : INFO : common : YAML file params.yaml loaded successfully.]
[2025-05-03 08:52:24,319 : INFO : common : Directory artifacts created successfully.]
[2025-05-03 08:52:24,321 : INFO : common : Directory artifacts/data_ingestion created successfully.]
[2025-05-03 08:52:26,989 : INFO : 1905418680 : artifacts/data_ingestion/data.zip downloaded with headers: 
 Connection: close
Content-Length: 16498017
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "1d0ebc96453e7b79c80e4d14e15a47bb3184c0c6f14b02e6c0ec2d81ab428c88"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: C1E3:2340ED:1FFD3B:5B032A:68158BF5
Accept-Ranges: bytes
Date: Sat, 03 May 2025 03:22:30 GMT
Via: 1.1 varnish
X-Served-By